# 03 - Variational Inference: KL Divergence and ELBO

In the previous notebook, MCMC approximated a posterior by drawing samples.
Variational Inference (VI) takes a different approach: it approximates the posterior
with a simpler distribution and turns inference into an optimization problem.

This notebook covers:
1. Why VI is useful
2. KL divergence as a measure of mismatch
3. Why the ELBO appears
4. A simple 1D optimization example

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## Part 1: Why Variational Inference?

Suppose the true posterior $p(\theta \mid x)$ is complicated.
Instead of sampling from it directly, VI chooses a simpler family of distributions
like $q(\theta)$ and asks:

**Which member of this family is closest to the true posterior?**

That turns inference into optimization.

## Part 2: KL divergence

A standard way to compare two distributions is the Kullback-Leibler divergence:
$$
athrm{KL}(q(\theta) \| p(\theta \mid x)) = nt q(\theta) \log \frac{q(\theta)}{p(\theta \mid x)} \, d\theta
$$

Important facts:
- $\mathrm{KL}(q \| p) \ge 0$
- It equals $0$ only if $q = p$ almost everywhere
- It is **not symmetric**: $\mathrm{KL}(q \| p) \neq \mathrm{KL}(p \| q)$

In [ ]:
x = np.linspace(-7, 7, 1000)

# A skewed / asymmetric target posterior-like density
p = 0.55 * stats.norm.pdf(x, loc=-1.5, scale=0.8) + 0.45 * stats.norm.pdf(x, loc=1.8, scale=1.6)
p = p / np.trapz(p, x)

# Candidate approximations q
q_good = stats.norm.pdf(x, loc=0.0, scale=2.0)
q_good = q_good / np.trapz(q_good, x)

q_bad = stats.norm.pdf(x, loc=2.6, scale=0.6)
q_bad = q_bad / np.trapz(q_bad, x)

plt.figure(figsize=(10, 4))
plt.plot(x, p, label='target posterior p(theta|x)', linewidth=3, color='navy')
plt.plot(x, q_good, label='candidate q: broad Gaussian', linewidth=2, color='darkorange')
plt.plot(x, q_bad, label='candidate q: narrow shifted Gaussian', linewidth=2, color='crimson')
plt.xlabel('theta')
plt.ylabel('density')
plt.title('A posterior and two variational candidates')
plt.legend()
plt.show()

In [ ]:
def discrete_kl(q, p, eps=1e-12):
    q_safe = np.clip(q, eps, None)
    p_safe = np.clip(p, eps, None)
    return np.trapz(q_safe * np.log(q_safe / p_safe), x)

kl_good = discrete_kl(q_good, p)
kl_bad = discrete_kl(q_bad, p)

print(f'KL(q_good || p) = {kl_good:.4f}')
print(f'KL(q_bad  || p) = {kl_bad:.4f}')
print('Smaller KL means a better approximation.')

## Part 3: Where the ELBO comes from

Start with the log evidence:
$$
og p(x)
$$
Insert any distribution $q(\theta)$:
$$
og p(x) = athcal{L}(q) + athrm{KL}(q(\theta) \| p(\theta \mid x))
$$
where
$$
athcal{L}(q) = athbb{E}_q[og p(x, \theta)] - athbb{E}_q[og q(\theta)]
$$
is the **Evidence Lower Bound** (ELBO).

Since KL is always nonnegative, the ELBO is a lower bound on $og p(x)$.
Maximizing the ELBO is equivalent to minimizing $\mathrm{KL}(q \| p)$.

In [ ]:
# Simple Bayesian model: theta ~ N(0, 1), x_i | theta ~ N(theta, sigma^2)
# We will pretend we do not know the exact posterior and optimize a Gaussian q(theta).

observed_x = np.array([1.2, 0.9, 1.4, 1.0, 1.1])
sigma_likelihood = 0.7

def log_joint(theta):
    log_prior = stats.norm.logpdf(theta, loc=0.0, scale=1.0)
    log_lik = np.sum(stats.norm.logpdf(observed_x, loc=theta, scale=sigma_likelihood), axis=0)
    return log_prior + log_lik

In [ ]:
def sample_q(mu, log_sigma, n_samples=4000):
    sigma = np.exp(log_sigma)
    return np.random.normal(loc=mu, scale=sigma, size=n_samples)

def estimate_elbo(params, n_samples=4000):
    mu, log_sigma = params
    sigma = np.exp(log_sigma)
    theta_samples = np.random.normal(mu, sigma, size=n_samples)

    log_p = log_joint(theta_samples)
    log_q = stats.norm.logpdf(theta_samples, loc=mu, scale=sigma)
    return np.mean(log_p - log_q)

def objective(params):
    # minimize negative ELBO
    return -estimate_elbo(params, n_samples=3000)

In [ ]:
initial_guess = np.array([0.0, np.log(1.5)])
result = minimize(objective, initial_guess, method='Nelder-Mead', options={'maxiter': 200, 'xatol': 1e-2, 'fatol': 1e-2})

mu_opt, log_sigma_opt = result.x
sigma_opt = np.exp(log_sigma_opt)

print('Optimization success:', result.success)
print(f'Optimal variational mean: {mu_opt:.3f}')
print(f'Optimal variational std:  {sigma_opt:.3f}')
print(f'Estimated ELBO: {-result.fun:.3f}')

In [ ]:
# Compute exact posterior here only for comparison and intuition
prior_var = 1.0
lik_var = sigma_likelihood**2
n = len(observed_x)

posterior_var = 1 / (1/prior_var + n/lik_var)
posterior_mean = posterior_var * (np.sum(observed_x) / lik_var)
posterior_std = np.sqrt(posterior_var)

theta_grid = np.linspace(-1, 2.5, 1000)
true_post = stats.norm.pdf(theta_grid, posterior_mean, posterior_std)
q_opt = stats.norm.pdf(theta_grid, mu_opt, sigma_opt)

plt.figure(figsize=(10, 4))
plt.plot(theta_grid, true_post, label='true posterior', linewidth=3, color='navy')
plt.plot(theta_grid, q_opt, label='optimized variational q', linewidth=2, color='darkorange')
plt.axvline(posterior_mean, color='navy', linestyle='--', alpha=0.6)
plt.axvline(mu_opt, color='darkorange', linestyle='--', alpha=0.6)
plt.xlabel('theta')
plt.ylabel('density')
plt.title('Exact posterior vs optimized variational approximation')
plt.legend()
plt.show()

print(f'True posterior mean: {posterior_mean:.3f}')
print(f'True posterior std:  {posterior_std:.3f}')

## Part 4: What this means intuitively

VI does not try to reproduce the posterior by random exploration like MCMC.
Instead, it picks a family $q$ and tunes its parameters to maximize the ELBO.

So the workflow is:
1. Choose a tractable family $q(\theta; \phi)$
2. Write down the ELBO
3. Optimize with respect to $\phi$
4. Use the optimized $q$ as the approximation

This is why VI is often much faster than MCMC in large models.

In [ ]:
# Optional: inspect how ELBO changes with q's mean while keeping std fixed
candidate_means = np.linspace(-0.5, 2.0, 50)
fixed_sigma = sigma_opt
elbos = []

for m in candidate_means:
    elbos.append(estimate_elbo([m, np.log(fixed_sigma)], n_samples=2000))

plt.figure(figsize=(8, 4))
plt.plot(candidate_means, elbos, color='purple', linewidth=2)
plt.axvline(mu_opt, color='darkorange', linestyle='--', label='optimized mean')
plt.xlabel('variational mean')
plt.ylabel('estimated ELBO')
plt.title('ELBO as a function of variational mean')
plt.legend()
plt.show()

## Summary

What to remember:
1. VI approximates the posterior with a simpler family $q$
2. KL divergence measures how far $q$ is from the posterior
3. ELBO maximization is the same as minimizing $\mathrm{KL}(q \| p)$
4. VI turns inference into optimization

This is the core conceptual move behind modern variational inference.

In [ ]:
# Exercises
# 1) Change observed_x to be more spread out. What happens to the posterior and q?
# 2) Try a worse initial guess for the optimizer. Does it still recover a good q?
# 3) Replace q with a broader Gaussian by hand and compare the KL numerically.

pass